# Annotation inspection analysis

Load pair-level summary and per-cell UMAP parquet from `run_annotation_inspection_pipeline.py`, then explore cytescore vs confidence and plot lightweight UMAPs without reloading h5ad files.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import scanpy as sc
import seaborn as sns
import pandas as pd

from shared.repo import REPO_ROOT

FIGS_DIR = REPO_ROOT / "writeups/state_vs_cytetype/.figs"
FIGS_DIR.mkdir(exist_ok=True)

In [ ]:
RUN_DIR = REPO_ROOT / "output/annotation_inspection_pipeline" / "20260605_223619"

summary = pd.read_csv(RUN_DIR / "summary.csv")
extremes = pd.read_csv(RUN_DIR / "extremes.csv")

summary["pair_label"] = summary["cell_type"] + summary["cytetype_annotation_leiden_merged"]
summary

In [ ]:
print(f"n accessions: {summary["accession"].nunique()}")
print(f"n unique STATE labels: {summary["cell_type"].nunique()}")
print(f"n unique CyteType annotations: {summary["cytetype_annotation_leiden_merged"].nunique()}")

In [ ]:
grp_state = summary.groupby("cell_type")
grp_cyte = summary.groupby("cytetype_annotation_leiden_merged")

# TODO: normalize by n_cells, either by number of occurrences of a given label pair, or by number of occurrances of a STATE label, etc.
cell_type_order = grp_state["cytescore_similarity"].median().sort_values(ascending=False).index

fig, ax = plt.subplots()
sns.boxplot(
    data=summary,
    x="cytescore_similarity",
    y="cell_type",
    order=cell_type_order,
    fliersize=0,
    ax=ax,
)

ax.set_title("Per-STATE label CyteScore distributions\nacross STATE label-CyteType annotation pairs")
ax.set_xlabel("CyteScore")
ax.set_ylabel("STATE label")
plt.tight_layout()
fig.savefig(FIGS_DIR / "cytescore_by_state_boxplot.png", dpi=150, bbox_inches="tight")

In [ ]:
x = list(grp_lo["cytescore_similarity"].mean().sort_values().index)

x.index("neutrophil")

In [ ]:
DELTA_COLS = ("low -> moderate", "moderate -> high")
DELTA_COLORS = ("#FFA500", "#228B22")
LEGEND_LABELS = (r"low $\rightarrow$ moderate", r"moderate $\rightarrow$ high")


def mean_rank(confidence: str) -> dict[str, int]:
    mask = summary["cytetype_confidence"] == confidence
    subset = summary.loc[mask]
    weighted_mean = (
        # multiply cytescore for a label pair by the number of cells with that label pair
        subset.assign(_w=lambda df: df["cytescore_similarity"] * df["n_cells"])
        # add the weighted cytescore and the number of cells with that label pair
        .groupby("cell_type", observed=True)[["_w", "n_cells"]]
        .sum()
        # divide the weighted cytescore by the number of cells with that label pair
        .assign(s_bar=lambda df: df["_w"] / df["n_cells"])["s_bar"]
        # create sorted ranking
        .sort_values(ascending=False)
    )
    return dict(zip(weighted_mean.index, range(len(weighted_mean)), strict=False))


lo_rank = mean_rank("Low")
mo_rank = mean_rank("Moderate")
hi_rank = mean_rank("High")

delta = (
    pd.DataFrame(
        {
            "low -> moderate": [lo_rank[ct] - mo_rank[ct] for ct in lo_rank],
            "moderate -> high": [mo_rank[ct] - hi_rank[ct] for ct in lo_rank],
        },
        index=lo_rank.keys(),
    )
    .assign(total=lambda df: df.sum(axis=1))
    .sort_values("total", ascending=True)
    .drop(columns="total")
)

BAR_HEIGHT = 0.38

fig, ax = plt.subplots()#figsize=(10, 9))
y_positions = {ct: i for i, ct in enumerate(delta.index)}

for ct in delta.index:
    y = y_positions[ct]
    lo_mo = delta.loc[ct, "low -> moderate"]
    mo_hi = delta.loc[ct, "moderate -> high"]

    ax.barh(
        y + BAR_HEIGHT / 2,
        lo_mo,
        height=BAR_HEIGHT,
        color=DELTA_COLORS[0],
        label=LEGEND_LABELS[0] if ct == delta.index[0] else None,
    )
    ax.barh(
        y - BAR_HEIGHT / 2,
        mo_hi,
        left=lo_mo,
        height=BAR_HEIGHT,
        color=DELTA_COLORS[1],
        label=LEGEND_LABELS[1] if ct == delta.index[0] else None,
    )

ax.axvline(0, color="grey", linewidth=0.5, zorder=0)
ax.set_yticks(list(y_positions.values()))
ax.set_yticklabels(list(y_positions.keys()))
ax.set_xlabel(r"$\Delta$ " + "rank by weighted mean CyteScore\nacross CyteType annotations")
ax.set_ylabel("STATE label")
ax.grid(axis="y")
ax.legend(title="CyteType confidence\nchange bin")
plt.tight_layout()
fig.savefig(FIGS_DIR / "cytescore_rank_delta_by_confidence.png", dpi=150, bbox_inches="tight")
plt.show()

## Absolute (non-relative) views of CyteScore vs confidence

The rank-delta above is relative. These cells add the underlying CyteScore levels,
the absolute change across confidence bands, and a signed monotonicity statistic.

In [ ]:
CONF_ORDER = ["Low", "Moderate", "High"]
CONF_ORDINAL = {"Low": 0, "Moderate": 1, "High": 2}


def weighted_mean(confidence: str) -> pd.Series:
    """n_cells-weighted mean CyteScore per STATE label within one confidence band."""
    subset = summary.loc[summary["cytetype_confidence"] == confidence]
    return (
        subset.assign(_w=lambda df: df["cytescore_similarity"] * df["n_cells"])
        .groupby("cell_type", observed=True)[["_w", "n_cells"]]
        .sum()
        .assign(s_bar=lambda df: df["_w"] / df["n_cells"])["s_bar"]
    )


# index = STATE label, columns = confidence bands; NaN where a label is absent in a band
s_bar = pd.DataFrame({c: weighted_mean(c) for c in CONF_ORDER})

In [ ]:
delta_abs = (
    pd.DataFrame(
        {
            "low -> moderate": s_bar["Moderate"] - s_bar["Low"],
            "moderate -> high": s_bar["High"] - s_bar["Moderate"],
        }
    )
    .dropna()
    .assign(total=lambda df: df.sum(axis=1))
    .sort_values("total")
    .drop(columns="total")
)

BAR_HEIGHT = 0.38
X_PAD = 0.05
fig, ax = plt.subplots()
y_positions = {ct: i for i, ct in enumerate(delta_abs.index)}
for ct in delta_abs.index:
    y = y_positions[ct]
    ax.barh(
        y + BAR_HEIGHT / 2,
        delta_abs.loc[ct, "low -> moderate"],
        height=BAR_HEIGHT,
        color=DELTA_COLORS[0],
        label=LEGEND_LABELS[0] if ct == delta_abs.index[0] else None,
    )
    ax.barh(
        y - BAR_HEIGHT / 2,
        delta_abs.loc[ct, "moderate -> high"],
        left=delta_abs.loc[ct, "low -> moderate"],
        height=BAR_HEIGHT,
        color=DELTA_COLORS[1],
        label=LEGEND_LABELS[1] if ct == delta_abs.index[0] else None,
    )

ax.axvline(0, color="grey", linewidth=0.5, zorder=0)
ax.set_yticks(list(y_positions.values()))
ax.set_yticklabels(list(y_positions.keys()))
ax.set_xlabel(r"$\Delta$ weighted mean CyteScore")
ax.set_ylabel("STATE label")
ax.set_xlim(
    min(
        delta_abs.min().min(),
        (delta_abs["low -> moderate"] + delta_abs["moderate -> high"]).min()
    ) - X_PAD,
    max(
        delta_abs.max().max(),
        (delta_abs["low -> moderate"] + delta_abs["moderate -> high"]).max()
    ) + X_PAD,
)
ax.grid(axis="y")
ax.legend(title="CyteType confidence\nchange bin")
plt.tight_layout()
fig.savefig(FIGS_DIR / "cytescore_abs_delta_by_confidence.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from scipy.stats import spearmanr

# global, pooled pair-level rows (unweighted by n_cells, by design)
conf_ord_all = summary["cytetype_confidence"].map(CONF_ORDINAL)
rho_all, p_all = spearmanr(conf_ord_all, summary["cytescore_similarity"])
print(f"global Spearman rho={rho_all:.3f}, p={p_all:.2e}")

rows = []
for i, (ct, g) in enumerate(summary.groupby("cell_type", observed=True)):
    conf_ord = g["cytetype_confidence"].map(CONF_ORDINAL)
    if conf_ord.nunique() < 2:  # need at least two bands to define a trend
        continue
    rho, pval = spearmanr(conf_ord, g["cytescore_similarity"])
    # if pval < 0.05:
    #     fig, ax = plt.subplots()
    #     ax.scatter(g["cytetype_confidence"], g["cytescore_similarity"], alpha=0.2)
    #     ax.invert_xaxis()
   
    rows.append({"cell_type": ct, "rho": rho, "pval": pval, "n_pairs": len(g)})

spear = (
    pd.DataFrame(rows)
    .dropna(subset=["rho"])
    .set_index("cell_type")
    .sort_values("rho")
)

colors = [
    "#228B22" if (p < 0.05 and r > 0) else "#FFA500" if (p < 0.05 and r < 0) else "lightgrey"
    for r, p in zip(spear["rho"], spear["pval"])
]

fig, ax = plt.subplots()
ax.barh(spear.index, spear["rho"], color=colors)
ax.axvline(0, color="grey", linewidth=0.5)
ax.set_xlabel(r"Spearman $\rho$ (CyteScore vs confidence)")
ax.set_ylabel("STATE label")
ax.set_title(f"global " + rf"$\rho$={rho_all:.2f} (p={p_all:.1e}); green/orange = p<0.05")
plt.tight_layout()
fig.savefig(FIGS_DIR / "cytescore_confidence_spearman.png", dpi=150, bbox_inches="tight")
plt.show()

### Weighted spearmans rank correlation

In [ ]:
import numpy as np
from scipy.stats import rankdata


def weighted_spearman(x, y, w):
    """n_cells-weighted Spearman: weighted Pearson on average ranks."""
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    w = np.asarray(w, float)
    m = np.isfinite(x) & np.isfinite(y) & np.isfinite(w)
    x, y, w = x[m], y[m], w[m]
    if len(x) < 2 or np.unique(x).size < 2 or np.unique(y).size < 2:
        return np.nan
    rx, ry = rankdata(x), rankdata(y)
    mx = np.average(rx, weights=w)
    my = np.average(ry, weights=w)
    cov = np.average((rx - mx) * (ry - my), weights=w)
    vx = np.average((rx - mx) ** 2, weights=w)
    vy = np.average((ry - my) ** 2, weights=w)
    return cov / np.sqrt(vx * vy)


def weighted_spearman_ci(x, y, w, n_boot=2000, seed=0):
    """Pair-level bootstrap 95% CI for the weighted Spearman rho."""
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    w = np.asarray(w, float)
    rng = np.random.default_rng(seed)
    n = len(x)
    boots = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        boots[b] = weighted_spearman(x[idx], y[idx], w[idx])
    boots = boots[np.isfinite(boots)]
    return np.percentile(boots, [2.5, 97.5])


# global, pooled pair-level rows weighted by n_cells
conf_ord_all = summary["cytetype_confidence"].map(CONF_ORDINAL)
rho_all_w = weighted_spearman(
    conf_ord_all, summary["cytescore_similarity"], summary["n_cells"]
)
lo_all, hi_all = weighted_spearman_ci(
    conf_ord_all.to_numpy(),
    summary["cytescore_similarity"].to_numpy(),
    summary["n_cells"].to_numpy(),
)
print(f"global weighted Spearman rho={rho_all_w:.3f}, 95% CI [{lo_all:.3f}, {hi_all:.3f}]")

rows = []
for ct, g in summary.groupby("cell_type", observed=True):
    conf_ord = g["cytetype_confidence"].map(CONF_ORDINAL)
    if conf_ord.nunique() < 2:  # need at least two bands to define a trend
        continue
    x = conf_ord.to_numpy()
    y = g["cytescore_similarity"].to_numpy()
    w = g["n_cells"].to_numpy()
    rho = weighted_spearman(x, y, w)
    lo, hi = weighted_spearman_ci(x, y, w)
    rows.append(
        {"cell_type": ct, "rho": rho, "ci_lo": lo, "ci_hi": hi, "n_pairs": len(g)}
    )

spear_w = (
    pd.DataFrame(rows)
    .dropna(subset=["rho"])
    .set_index("cell_type")
    .sort_values("rho")
)

# significance from whether the bootstrap CI excludes zero
colors = [
    "#228B22" if lo > 0 else "#FFA500" if hi < 0 else "lightgrey"
    for lo, hi in zip(spear_w["ci_lo"], spear_w["ci_hi"])
]

fig, ax = plt.subplots()
ax.barh(spear_w.index, spear_w["rho"], color=colors)
ax.axvline(0, color="grey", linewidth=0.5)
ax.set_xlabel(r"Weighted Spearman $\rho$ (CyteScore vs confidence)")
ax.set_ylabel("STATE label")
ax.set_title(
    "global weighted " + rf"$\rho$={rho_all_w:.2f} (95% CI [{lo_all:.2f}, {hi_all:.2f}]); green/orange = CI excludes 0"
)
plt.tight_layout()
fig.savefig(
    FIGS_DIR / "cytescore_confidence_spearman_weighted.png", dpi=150, bbox_inches="tight"
)
plt.show()

In [ ]:
from scipy.stats import spearmanr
import numpy as np

# global trend used as the chart annotation (the "simplest stat" on this question)
conf_ord_all = summary["cytetype_confidence"].map(CONF_ORDINAL)
rho_all, p_all = spearmanr(conf_ord_all, summary["cytescore_similarity"])

x = [CONF_ORDINAL[c] for c in CONF_ORDER]

fig, ax = plt.subplots(figsize=(5, 5))

last_position = 2
n = len(s_bar)
cmap = plt.get_cmap("hsv")  # or "plasma", "coolwarm", etc.

for i, (ct, row) in enumerate(s_bar.sort_values("High", ascending=False).iterrows()):
    # if ct in HIGHLIGHT:
    ax.plot(
        x,
        row[CONF_ORDER].to_numpy(),
        marker="o",
        linewidth=2.0,
        color=cmap(i / max(n - 1, 1)),
        label=ct,
    )
    if last_position - row[CONF_ORDER].to_numpy()[-1] > 0.02:
        ax.text(2.15, row[CONF_ORDER].to_numpy()[-1], ct)
        last_position = row[CONF_ORDER].to_numpy()[-1]
    # else:
    #     ax.text(3.5, row[CONF_ORDER].to_numpy()[-1], ct)

ax.set_xticks(x)
ax.set_xticklabels(CONF_ORDER)
ax.set_xlabel("CyteType confidence")
ax.set_ylabel("weighted mean CyteScore")
# ax.set_title(f"global Spearman " + rf"$\rho$={rho_all:.2f} (p={p_all:.1e})")
# ax.legend(HIGHLIGHT, title="STATE label\n(top and bottom 3 largest delta shown)")
plt.tight_layout()
fig.savefig(FIGS_DIR / "cytescore_level_by_confidence.png", dpi=150, bbox_inches="tight")
plt.show()
ax.plot

In [ ]:
import statsmodels.formula.api as smf

model_df = summary[["cytescore_similarity", "cytetype_confidence", "accession"]].copy()
model_df["confidence_ordinal"] = model_df["cytetype_confidence"].map(CONF_ORDINAL)

# random intercept per accession; confidence_ordinal treated as a linear 0/1/2 predictor
mixed = smf.mixedlm(
    "cytescore_similarity ~ confidence_ordinal",
    data=model_df,
    groups=model_df["accession"],
)
result = mixed.fit()
print(result.summary())

slope = result.params["confidence_ordinal"]
pval = result.pvalues["confidence_ordinal"]
print(f"CyteScore change per confidence step = {slope:.4f} (p={pval:.2e})")

## Lightweight UMAP plots

In [ ]:
import json
import os
from pathlib import Path

from scanpy.plotting._utils import set_colors_for_categorical_obs

adata = sc.read(Path("..").resolve().parent / "data/cytetype_annotated/SRX23724122_annotated.h5ad")
accession = "SRX23724122"

cyteonto_df = pd.read_csv(
    REPO_ROOT / "output/cyteonto_pipeline/deduplicated_tables/deduplicated.csv"
)
cyteonto_df = cyteonto_df[cyteonto_df["accession"] == accession]
adata.obs["pair_label"] = (
    adata.obs["cell_type"].astype(str) + adata.obs["cytetype_annotation_leiden_merged"].astype(str)
)
cyteonto_df = cyteonto_df.set_index("pair_label")
adata.obs = adata.obs.set_index("pair_label")
adata.obs = adata.obs.join(cyteonto_df[["cytescore_similarity"]], how="left")
adata.obs = adata.obs.reset_index()

cluster_key = "leiden_merged"
payload = adata.uns["cytetype_results"]["result"]
cytetype_result = json.loads(payload) if isinstance(payload, str) else payload
confidence_by_cluster = {
    str(cluster_id): entry["latest"]["review"]["confidence"]
    for cluster_id, entry in cytetype_result["raw_annotations"].items()
}
adata.obs["cytetype_confidence"] = pd.Categorical(
    adata.obs[cluster_key].map(confidence_by_cluster),
    categories=["High", "Moderate", "Low"],
    ordered=True,
)

confidence_palette = {"Low": "#d73027", "Moderate": "#fee08b", "High": "#1a9850"}
set_colors_for_categorical_obs(adata, "cytetype_confidence", confidence_palette)

fig = sc.pl.umap(
    adata,
    color=[
        "cell_type",
        "cytetype_annotation_leiden_merged",
        "cytescore_similarity",
        "cytetype_confidence",
    ],
    title=[
        f"{accession}: STATE labels",
        f"{accession}: CyteType annotations",
        f"{accession}: CyteOnto cytescore similarity",
        f"{accession}: CyteType cluster confidence",
    ],
    ncols=2,
    legend_loc="on data",
    size=30,
    wspace=0.4,
    return_fig=True,
)
fig.savefig(FIGS_DIR / "umap_example_accession.png", dpi=50, bbox_inches="tight")